In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Конфигурация сетки 2x2 ---
# 12 отрезков (индексы 0..11), порядок фиксирован
EDGES = [
    ((0, 0), (1, 0)),  # 0: нижний левый горизонтальный
    ((1, 0), (2, 0)),  # 1: нижний правый горизонтальный
    ((0, 1), (1, 1)),  # 2: средний левый горизонтальный (внутр. левый)
    ((1, 1), (2, 1)),  # 3: средний правый горизонтальный (внутр. правый)
    ((0, 2), (1, 2)),  # 4: верхний левый горизонтальный
    ((1, 2), (2, 2)),  # 5: верхний правый горизонтальный
    ((0, 0), (0, 1)),  # 6: левый нижний вертикальный
    ((0, 1), (0, 2)),  # 7: левый верхний вертикальный
    ((1, 0), (1, 1)),  # 8: средний нижний вертикальный (внутр. нижний)
    ((1, 1), (1, 2)),  # 9: средний верхний вертикальный (внутр. верхний)
    ((2, 0), (2, 1)),  # 10: правый нижний вертикальный
    ((2, 1), (2, 2)),  # 11: правый верхний вертикальный
]


# Вектор направления для каждого отрезка: dx,dy = 2*(середина - центр(1,1))
def _midpoint(p1, p2):
    return ((p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2)


VECTORS = []
for p1, p2 in EDGES:
    mx, my = _midpoint(p1, p2)
    dx = int(2 * (mx - 1))
    dy = int(2 * (my - 1))
    VECTORS.append((dx, dy))

# Построение графа смежности отрезков (общие концы)
adj = {i: set() for i in range(12)}
for i, (a1, a2) in enumerate(EDGES):
    for j, (b1, b2) in enumerate(EDGES):
        if i >= j:
            continue
        if a1 == b1 or a1 == b2 or a2 == b1 or a2 == b2:
            adj[i].add(j)
            adj[j].add(i)


# Кратчайшие расстояния в графе отрезков (матрица 12x12)
def bfs_dist(start):
    dist = [None] * 12
    dist[start] = 0
    q = [start]
    for u in q:
        for v in adj[u]:
            if dist[v] is None:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist


DIST_MATRIX = np.array([bfs_dist(i) for i in range(12)], dtype=int)
MAX_DIST = np.max(DIST_MATRIX)  # = 3


# --- Симметрии квадрата (D4) ---
def sym_identity(v):
    return (v[0], v[1])


def sym_rot90(v):
    return (v[1], -v[0])


def sym_rot180(v):
    return (-v[0], -v[1])


def sym_rot270(v):
    return (-v[1], v[0])


def sym_flip_x(v):
    return (-v[0], v[1])


def sym_flip_y(v):
    return (v[0], -v[1])


def sym_flip_d1(v):
    return (v[1], v[0])


def sym_flip_d2(v):
    return (-v[1], -v[0])


SYMMETRIES = [
    sym_identity,
    sym_rot90,
    sym_rot180,
    sym_rot270,
    sym_flip_x,
    sym_flip_y,
    sym_flip_d1,
    sym_flip_d2,
]
SYM_NAMES = ["id", "rot90", "rot180", "rot270", "flipX", "flipY", "flipD1", "flipD2"]
COST = [0, 1, 1, 1, 1, 1, 1, 1]  # штраф за нетождественную симметрию

# Сопоставление индексов отрезков при каждой симметрии
vec_to_idx = {v: i for i, v in enumerate(VECTORS)}
SYMMETRY_MAPPING = []
for f in SYMMETRIES:
    mapping = {}
    for i, v in enumerate(VECTORS):
        new_v = f(v)
        mapping[i] = vec_to_idx[new_v]
    SYMMETRY_MAPPING.append(mapping)

print("Константы загружены. Готово.")

In [ ]:
def decode(bit_string: str) -> set:
    """
    Принимает строку из 12 символов '0'/'1'.
    Возвращает множество индексов присутствующих отрезков (0..11).
    """
    if len(bit_string) != 12 or not all(ch in '01' for ch in bit_string):
        raise ValueError("Строка должна содержать ровно 12 символов 0/1")
    return {i for i, ch in enumerate(bit_string) if ch == '1'}

# Пример
example = '111111111111'
print(f"Фигура: {example} -> индексы: {decode(example)}")

In [ ]:
def visualize(bit_string: str, ax=None):
    """
    Визуализация сетки 2x2:
    сплошные линии для '1', пунктирные серые для '0'.
    Каждый отрезок подписан своим индексом (0..11).
    """
    present = decode(bit_string)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5,5))
    
    for i, ((x1,y1),(x2,y2)) in enumerate(EDGES):
        # Определяем цвет и стиль линии
        if i in present:
            color = 'black'
            linestyle = '-'
            linewidth = 3
        else:
            color = 'gray'
            linestyle = 'dotted'
            linewidth = 1
        
        # Рисуем линию
        ax.plot([x1,x2], [y1,y2], color=color, linestyle=linestyle, linewidth=linewidth)
        
        # Подписываем индекс отрезка
        # Вычисляем середину отрезка для размещения текста
        mx = (x1 + x2) / 2
        my = (y1 + y2) / 2
        
        # Небольшое смещение чтобы текст не накладывался на линию
        # Для горизонтальных линий смещаем по вертикали, для вертикальных - по горизонтали
        if y1 == y2:  # горизонтальная линия
            offset_x = 0
            offset_y = 0.12
        else:  # вертикальная линия
            offset_x = 0.12
            offset_y = 0
        
        # Цвет текста: синий для видимых, серый для пунктирных
        text_color = 'blue' if i in present else 'lightgray'
        
        ax.text(mx + offset_x, my + offset_y, str(i), 
                fontsize=10, ha='center', va='center', 
                color=text_color, fontweight='bold')
    
    ax.set_aspect('equal')
    ax.set_xlim(-0.3, 2.3)
    ax.set_ylim(-0.3, 2.3)
    ax.axis('off')
    # Убираем внешние рамки графика
    ax.set_frame_on(False)
    plt.show()

# Демонстрация
test_str = '101010101010'
print(f"Визуализация: {test_str}")
visualize(test_str)

In [ ]:
def h_measure(A_set: set, B_set: set) -> int:
    """
    Внутренняя метрика (без симметрий) на множествах индексов.
    Z = пересечение, D = симметрическая разность.
    Сумма (2 + расстояние от e до Z) для e in D.
    """
    Z = A_set & B_set
    D = A_set ^ B_set
    if not Z:
        return sum(2 + MAX_DIST for _ in D)
    z_list = list(Z)
    total = 0
    for e in D:
        d_min = min(DIST_MATRIX[e, f] for f in z_list)
        total += 2 + d_min
    return total

def compare(bit_str1: str, bit_str2: str) -> int:
    """
    Итоговый коэффициент непохожести.
    Минимум по 8 симметриям: cost(σ) + h(σ(A), B).
    """
    A = decode(bit_str1)
    B = decode(bit_str2)
    best = 10**9
    for sigma, mapping in enumerate(SYMMETRY_MAPPING):
        sigma_A = {mapping[i] for i in A}
        total = COST[sigma] + h_measure(sigma_A, B)
        if total < best:
            best = total
    return best

# Примеры
s1 = '111111111111'  # все отрезки
s2 = '111111111110'  # не хватает 0-го отрезка (нижний левый горизонтальный)
print(f"compare({s1}, {s2}) = {compare(s1, s2)}")

# Поворот
s3 = '001111110000'  # некоторый узор
s4 = '110000111100'  # предположительно повёрнутый
print(f"Сравнение возможных поворотов: {compare(s3, s4)}")

In [ ]:
def compare_benchmark(reference: str, samples: list, label: str = ""):
    """
    Сравнивает эталонную строку reference с каждым образцом из samples.
    Выводит результаты в читаемом формате.
    
    Параметры:
    - reference: эталонная строка из 12 символов '0'/'1'
    - samples: список строк-образцов для сравнения
    - label: опциональное описание образцов (для заголовка)
    """
    print("=" * 60)
    if label:
        print(f"СРАВНЕНИЕ: {reference} vs {label}")
    else:
        print(f"СРАВНЕНИЕ: {reference} vs образцы")
    print("=" * 60)
    
    for s in samples:
        d = compare(reference, s)
        print(f"  {reference} vs {s} -> коэффициент = {d}")
    print()


# =====================================================
# Вспомогательные генераторы строк
# =====================================================
def generate_single_one_strings():
    """Генерирует все строки с ровно одной '1'"""
    return [''.join(['1' if i == j else '0' for j in range(12)]) for i in range(12)]

def generate_single_zero_strings():
    """Генерирует все строки с ровно одним '0'"""
    return [''.join(['0' if i == j else '1' for j in range(12)]) for i in range(12)]

def generate_growing_ones():
    """Генерирует строки с нарастающими единицами"""
    return ['1' * i + '0' * (12 - i) for i in range(1, 13)]


# =====================================================
# Запуск сравнений
# =====================================================
one_one_zeros = '000000000000'
zeroes_one_one  = '111111111111'
single_ones = generate_single_one_strings()
single_zeros = generate_single_zero_strings()
growing = generate_growing_ones()

# 1. Все нули vs коды с одной '1'
compare_benchmark(one_one_zeros, single_ones, "коды с одной '1'")

# 3. Все нули vs коды с одним '0'
compare_benchmark(one_one_zeros, single_zeros, "коды с одним '0'")

# 6. Все нули vs нарастающие единицы
compare_benchmark(one_one_zeros, growing, "нарастающие единицы")


# 2. Все единицы vs коды с одной '1'
compare_benchmark(zeroes_one_one, single_ones, "коды с одной '1'")

# 4. Все единицы vs коды с одним '0'
compare_benchmark(zeroes_one_one, single_zeros, "коды с одним '0'")

# 5. Все единицы vs нарастающие единицы
compare_benchmark(zeroes_one_one, growing, "нарастающие единицы")


In [ ]:
import math
from itertools import combinations


def combinations_count(n, k):
    """Возвращает число сочетаний C(n, k)"""
    return math.factorial(n) // (math.factorial(k) * math.factorial(n - k))


def generate_all_combinations(n, k):
    """
    Генерирует все строки из n бит с ровно k единицами.
    Возвращает список строк.
    """
    result = []
    for ones_positions in combinations(range(n), k):
        s = ["0"] * n
        for pos in ones_positions:
            s[pos] = "1"
        result.append("".join(s))
    return result


def group_by_distance(reference, samples):
    """
    Группирует образцы по расстоянию (непохожести) до эталона.
    Возвращает dict: {расстояние: [список строк]}
    """
    groups = {}
    for s in samples:
        d = compare(reference, s)
        if d not in groups:
            groups[d] = []
        groups[d].append(s)
    return groups


def print_groups(groups, reference):
    """
    Выводит группы расстояний с полным списком комбинаций.
    """
    print(f"Группировка по расстоянию до '{reference}':")
    print("-" * 50)
    for dist in sorted(groups.keys()):
        combos = groups[dist]
        print(f"{dist}:")
        for c in combos:
            print(f"\t{c}")
        print()


# Проверка формул
print("Число сочетаний:")
print(f"  C(12, 2) = {combinations_count(12, 2)}")  # -> 66
print(f"  C(12, 3) = {combinations_count(12, 3)}")  # -> 220
print(f"  C(12, 4) = {combinations_count(12, 4)}")  # -> 495
print()

# =====================================================
# Комбинации с 2 единицами (66 штук)
samples_2 = generate_all_combinations(12, 2)
print(f"Сгенерировано комбинаций с 2 единицами: {len(samples_2)}")
print()

one_one_zeros = "110000000000"
groups_zeros = group_by_distance(one_one_zeros, samples_2)
print_groups(groups_zeros, one_one_zeros)

# =====================================================
# zeroes_one_one = "000000000011"
# groups_ones = group_by_distance(zeroes_one_one, samples_2)
# print_groups(groups_ones, zeroes_one_one)

In [ ]:
import matplotlib.pyplot as plt


def draw_combo(ax, bit_string, x_offset, y_offset, cell_size=0.6):
    """
    Рисует один квадрат 2×2 с линиями.
    x_offset, y_offset — координаты левого нижнего угла квадрата 2×2.
    cell_size — размер одной ячейки сетки.
    """
    present = decode(bit_string)

    # Масштабируем координаты отрезков
    for i, ((x1, y1), (x2, y2)) in enumerate(EDGES):
        # Смещаем координаты
        xx1 = x_offset + x1 * cell_size
        yy1 = y_offset + y1 * cell_size
        xx2 = x_offset + x2 * cell_size
        yy2 = y_offset + y2 * cell_size

        if i in present:
            ax.plot([xx1, xx2], [yy1, yy2], "k-", linewidth=1.5)
        else:
            ax.plot(
                [xx1, xx2], [yy1, yy2], color="gray", linestyle="dotted", linewidth=0.6
            )

    # Подпись: код комбинации под квадратом
    ax.text(
        x_offset + 1 * cell_size,
        y_offset - 0.15,
        bit_string,
        fontsize=9,
        ha="center",
        va="top",
        family="monospace",
    )


def visualize_groups(groups, reference, max_cols=10):
    """
    Для каждой группы расстояний создаёт отдельную фигуру.
    Все фигуры имеют одинаковый масштаб: max_cols столбцов.
    """
    # Фиксированные размеры ячейки (одинаковые для всех групп)
    cell_w = 1.8  # ширина одной ячейки с квадратом
    cell_h = 2.0  # высота одной ячейки с квадратом и подписью

    for dist in sorted(groups.keys()):
        combos = groups[dist]
        n = len(combos)

        # ВСЕГДА используем max_cols столбцов (даже если комбинаций меньше)
        cols = max_cols
        rows = (n + cols - 1) // cols
        if rows == 0:
            rows = 1

        fig_width = cols * cell_w + 0.5
        fig_height = rows * cell_h + 0.8

        fig, axes = plt.subplots(
            rows, cols, figsize=(fig_width, fig_height), squeeze=False
        )

        fig.suptitle(
            f"Расстояние: {dist}  |  Эталон: {reference}  |  Комбинаций: {n}",
            fontsize=12,
            fontweight="bold",
        )

        # Рисуем только нужные квадраты, остальные ячейки остаются пустыми
        for idx in range(n):
            r = idx // cols
            c = idx % cols
            ax = axes[r][c]

            # Рисуем квадрат в центре ячейки
            draw_combo(ax, combos[idx], x_offset=0.3, y_offset=0.2)

            # Настройка осей для ячейки
            ax.set_xlim(0, 1.8)
            ax.set_ylim(-0.3, 1.8)
            ax.set_aspect("equal")
            ax.axis("off")

        # Скрываем все неиспользуемые ячейки (включая пустые в последнем ряду)
        for idx in range(rows * cols):
            r = idx // cols
            c = idx % cols
            if idx >= n:
                axes[r][c].axis("off")
            # Для используемых ячеек оси уже настроены выше

        plt.tight_layout()
        plt.show()
        print()  # пустая строка между группами


# =====================================================
# Визуализация groups_zeros из ячейки 6
# =====================================================
print("Визуализация групп для эталона '110000000000'")
print("=" * 60)

# groups_zeros уже вычислен в ячейке 6
visualize_groups(groups_zeros, one_one_zeros, max_cols=10)

In [ ]:
# =====================================================
# Эталон и генерация всех комбинаций с 3 единицами
# =====================================================
reference_3 = "110000001000"
samples_3 = generate_all_combinations(12, 3)
print(f"Сгенерировано комбинаций с 3 единицами: {len(samples_3)}")
print()

# =====================================================
# Группировка по расстоянию до эталона
# =====================================================
groups_3 = group_by_distance(reference_3, samples_3)

# =====================================================
# Вывод текстовой группировки
# =====================================================
print_groups(groups_3, reference_3)

# =====================================================
# Визуализация групп
# =====================================================
print("Визуализация групп для эталона '110000001000'")
print("=" * 60)
visualize_groups(groups_3, reference_3, max_cols=10)